# Synthetic data check

**Question:** do the synthetic datasets load, and do they look like the assumptions in
`scripts/make_data.py` say they should?

**Data assumptions:** all data are synthetic, copper (Cu) in water by ICP-OES, generated by
`scripts/make_data.py` with a fixed seed. Lab bias +1.2 %, within-lab reproducibility 3 % RSD.
No real laboratory data.

This notebook also proves the build pipeline: CI executes it and fails if any cell errors.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA = Path("..") / "data" / "synthetic"
if not DATA.exists():  # run from the repository root
    DATA = Path("data") / "synthetic"

blanks = pd.read_csv(DATA / "blanks.csv")
calibration = pd.read_csv(DATA / "calibration.csv")
qc = pd.read_csv(DATA / "qc.csv", parse_dates=["date"])
pt = pd.read_csv(DATA / "pt.csv", parse_dates=["date"])

for name, frame in {"blanks": blanks, "calibration": calibration, "qc": qc, "pt": pt}.items():
    print(f"{name:12s} {frame.shape[0]:3d} rows  columns: {', '.join(frame.columns)}")

## Do the summary numbers match the stated assumptions?

In [ ]:
qc_bias_pct = 100 * (qc["cu_ug_L"].mean() / qc["target_ug_L"].mean() - 1)
qc_rsd_pct = 100 * qc["cu_ug_L"].std(ddof=1) / qc["cu_ug_L"].mean()
pt_bias_pct = 100 * (pt["lab_result_ug_L"] / pt["x_pt_ug_L"] - 1).mean()

summary = pd.Series(
    {
        "Blank mean (ug/L), assumed 0.40": blanks["cu_ug_L"].mean(),
        "Blank SD (ug/L), assumed about 0.27": blanks["cu_ug_L"].std(ddof=1),
        "QC bias (%), assumed +1.2": qc_bias_pct,
        "QC RSD (%), assumed 3": qc_rsd_pct,
        "PT mean bias (%), assumed +1.2": pt_bias_pct,
    }
).round(2)
summary

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(calibration["cu_ug_L"], calibration["intensity_cps"] / 1000, s=18)
slope, intercept = np.polyfit(calibration["cu_ug_L"], calibration["intensity_cps"], 1)
x = np.linspace(0, calibration["cu_ug_L"].max(), 50)
ax.plot(x, (intercept + slope * x) / 1000, lw=1)
ax.set_xlabel("Cu (ug/L)")
ax.set_ylabel("Intensity (kcps)")
ax.set_title(f"Calibration: slope {slope:.0f} cps per ug/L (assumed 1200)")
plt.show()